# JSON Parsing and Processing

JSON is one of the most common formats for storing structured data.

It can be simple, but real-world JSON can also become deeply nested with lists, objects, and multiple levels of information.

In this notebook, I’m learning how to load JSON data, work with nested structures, and convert useful parts of the JSON into LangChain `Document` objects.

I’m also looking at JSONL because it is another common format used for logs, events, and large datasets.

The main flow I’m following is:

**JSON / JSONL → Extract Useful Data → Create Documents → Preserve Context and Metadata**


## Setting Up the JSON Environment

I’m importing the basic Python libraries first and creating a folder where I can keep the JSON examples.

I’m using sample data here so I can experiment with different JSON structures without depending on external files.


In [1]:

import json
import os

os.makedirs("E:/RAG/data/course_samples/json_files", exist_ok=True)

## Creating a Nested JSON Example

I’m creating a small company dataset with nested information.

The JSON contains:

- Company information
- Employee details
- Skills
- Projects
- Department information

I’m keeping it nested on purpose because real-world JSON is often not just a flat list of values.


In [3]:
json_data = {
    "company": "TechCorp",

    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"}
            ]
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {"name": "ML Model", "status": "In Progress"},
                {"name": "Analytics Dashboard", "status": "Planning"}
            ]
        }
    ],

    "departments": {
        "engineering": {
            "head": "Mike Johnson",
            "budget": 1000000,
            "team_size": 25
        },
        "data_science": {
            "head": "Sarah Williams",
            "budget": 750000,
            "team_size": 15
        }
    }
}

## Looking at the JSON Structure

Before saving or processing the data, I want to look at the Python object itself.

This helps me understand how the nested JSON is represented after loading it into Python.


In [4]:
json_data

{'company': 'TechCorp',
 'employees': [{'id': 1,
   'name': 'John Doe',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'React'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Jane Smith',
   'role': 'Data Scientist',
   'skills': ['Python', 'Machine Learning', 'SQL'],
   'projects': [{'name': 'ML Model', 'status': 'In Progress'},
    {'name': 'Analytics Dashboard', 'status': 'Planning'}]}],
 'departments': {'engineering': {'head': 'Mike Johnson',
   'budget': 1000000,
   'team_size': 25},
  'data_science': {'head': 'Sarah Williams',
   'budget': 750000,
   'team_size': 15}}}

### What I noticed

The data contains different levels of nesting.

For example, `employees` is a list of employee objects, and each employee also contains a list of projects.

This is where JSON processing becomes more interesting because I may not want to treat the whole file as one document.


## Saving the JSON File

Now I’m saving the sample data as an actual `.json` file.

This gives me a real file to test with the different JSON loaders below.


In [5]:
with open("E:/RAG/data/course_samples/json_files/company_data.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=2)

print("JSON file created.")

JSON file created.


## Creating a JSONL File

JSONL stands for JSON Lines.

Instead of storing one large JSON structure, each line contains one separate JSON object.

This format is useful for things like:

- Event logs
- User activity
- Transactions
- Streaming data
- Large datasets

I’m creating a small example so I can see the difference between normal JSON and JSONL.


In [6]:
jsonl_data = [
    {
        "timestamp": "2024-01-01",
        "event": "user_login",
        "user_id": 123
    },
    {
        "timestamp": "2024-01-01",
        "event": "page_view",
        "user_id": 123,
        "page": "/home"
    },
    {
        "timestamp": "2024-01-01",
        "event": "purchase",
        "user_id": 123,
        "amount": 99.99
    }
]

with open("E:/RAG/data/course_samples/json_files/events.jsonl", "w", encoding="utf-8") as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + "\n")

print("JSONL file created.")

JSONL file created.


## JSON Processing Strategies

Now I want to try two different approaches for processing the JSON data.

First, I’ll use LangChain's `JSONLoader` with a `jq_schema` to select specific parts of the JSON.

Then I’ll build my own processing function where I have more control over how the nested data is converted into documents.

The goal is not just to read the JSON.

I want to understand how the structure of the JSON affects the documents I create for RAG.


## 1. Using JSONLoader with `jq_schema`

The first approach is using LangChain's `JSONLoader`.

I’m using `jq_schema` to select the `employees` list and create one document for each employee.

This is useful when I already know which part of the JSON I want to retrieve.


In [10]:
from langchain_community.document_loaders import JSONLoader

# MEthod1 : JsonLoader With jq_schema
print("1️⃣ JSONLoader - Extract specific fields")

# Extract employee information
employee_loader = JSONLoader(
    file_path="E:/RAG/data/course_samples/json_files/company_data.json",
    jq_schema=".employees[]",     # jq query to extract each employee
    text_content=False            # Get full JSON objects
)

employee_docs = employee_loader.load()

print(f"Loaded {len(employee_docs)} employee documents")
print(f"First employee: {employee_docs[0].page_content[:200]}...")
print(employee_docs)

1️⃣ JSONLoader - Extract specific fields
Loaded 2 employee documents
First employee: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status"...
[Document(metadata={'source': 'E:\\RAG\\data\\course_samples\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': 'E:\\RAG\\data\\course_samples\\json_files\\company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "Analytics Dashboard", "status": "Planning"}]}')]


### What I noticed

The `jq_schema` lets me select only the part of the JSON that I care about.

In this case:

```text
.employees[]
```


In [11]:
employee_docs

[Document(metadata={'source': 'E:\\RAG\\data\\course_samples\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'),
 Document(metadata={'source': 'E:\\RAG\\data\\course_samples\\json_files\\company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "Analytics Dashboard", "status": "Planning"}]}')]

---

# 9. Why `jq_schema` Matters

### Markdown Cell

```markdown
## Why `jq_schema` Is Useful

A JSON file can contain a lot of information that may not be relevant to a particular RAG use case.

For example, I may only want:

- Employee profiles
- Customer records
- Orders
- Product information
- Event data

Using `jq_schema`, I can select the exact part of the JSON that I want to turn into documents.

This gives me more control during ingestion.
```


## 2. Custom JSON Processing

Now I’m taking a different approach.

Instead of letting the loader decide how the JSON should be represented, I’m creating the documents myself.

This gives me control over:

- What information goes into the document
- How nested fields are flattened
- Which metadata is stored
- How much context is preserved

This becomes useful when the default JSON structure is not the best format for retrieval.


In [12]:
from typing import List
from langchain_core.documents import Document

## Creating a Custom JSON Processor

I’m creating one document for each employee.

For every employee, I’m keeping:

- Name
- Role
- Skills
- Projects

I’m also storing employee information in metadata so I can use it later for filtering, tracing, or debugging.


In [13]:
# Method 2: Custom JSON processing for complex structures
print("\n2️⃣ Custom JSON Processing")


def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with context preserved for each employee."""

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents = []

    # Strategy 1: Create documents for each employee with full context
    for emp in data.get("employees", []):

        content = f"""Employee Profile:
        Name: {emp['name']}
        Role: {emp['role']}
        Skills: {', '.join(emp['skills'])}

        Projects:"""

        for proj in emp.get("projects", []):
            content += f"\n- {proj['name']} (Status: {proj['status']})"

        doc = Document(
            page_content=content,
            metadata={
                "source": filepath,
                "data_type": "employee_profile",
                "employee_id": emp["id"],
                "employee_name": emp["name"],
                "role": emp["role"]
            }
        )

        documents.append(doc)

    return documents


2️⃣ Custom JSON Processing


## Testing the Custom Processor

Now I’m running the custom processor on the same JSON file.

The main thing I want to check is whether each employee becomes a clean document while keeping the useful context from the nested JSON.


In [14]:
custom_docs = process_json_intelligently(
    "E:/RAG/data/course_samples/json_files/company_data.json"
)

print(f"Created {len(custom_docs)} documents")

Created 2 documents


## Looking at the Custom Documents

I’m printing the generated documents so I can compare the final text and metadata.

This helps me understand the difference between simply extracting JSON and actually preparing it for retrieval.


In [15]:
for i, doc in enumerate(custom_docs):

    print(f"\nDocument {i + 1}")
    print("-" * 40)

    print(doc.page_content)

    print("\nMetadata:")
    print(doc.metadata)


Document 1
----------------------------------------
Employee Profile:
        Name: John Doe
        Role: Software Engineer
        Skills: Python, JavaScript, React

        Projects:
- RAG System (Status: In Progress)
- Data Pipeline (Status: Completed)

Metadata:
{'source': 'E:/RAG/data/course_samples/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}

Document 2
----------------------------------------
Employee Profile:
        Name: Jane Smith
        Role: Data Scientist
        Skills: Python, Machine Learning, SQL

        Projects:
- ML Model (Status: In Progress)
- Analytics Dashboard (Status: Planning)

Metadata:
{'source': 'E:/RAG/data/course_samples/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}


### What I noticed

The custom approach gives me a cleaner format for retrieval.

Instead of storing the raw nested JSON, I converted it into a readable employee profile.

At the same time, I kept important fields in metadata.

This gives me both:

**Readable content for semantic search**

and

**Structured metadata for filtering and traceability**


## Comparing the Two Approaches

Now I want to compare the two ways of processing the same JSON.

`JSONLoader` is useful when I want a quick and flexible way to extract specific parts of a JSON structure.

Custom processing gives me more control over the final document format and metadata.

Neither approach needs to be used for every JSON file.

The right choice depends on the structure of the data and what I need to retrieve later.


In [16]:
print("JSON PROCESSING COMPARISON")

print("\nJSONLoader:")
print(f"  Documents created: {len(employee_docs)}")
print("  Uses jq_schema to select data")
print("  Good for targeted extraction")

print("\nCustom JSON Processing:")
print(f"  Documents created: {len(custom_docs)}")
print("  Custom document structure")
print("  Rich metadata")
print("  More control over nested data")

JSON PROCESSING COMPARISON

JSONLoader:
  Documents created: 2
  Uses jq_schema to select data
  Good for targeted extraction

Custom JSON Processing:
  Documents created: 2
  Custom document structure
  Rich metadata
  More control over nested data


## Why This Matters in RAG

Imagine I have a company knowledge base stored as JSON.

A user might ask:

- Who is the Data Scientist?
- What skills does John Doe have?
- Which projects is John working on?
- Which employees know Python?
- What is Jane's current project status?

I could create one huge document from the entire JSON file, but that would mix information from different employees.

Instead, I can create separate documents for each employee and keep employee-specific information in metadata.

This gives the retriever smaller and more focused pieces of information to work with.


## My Takeaway

JSON can look simple at first, but nested structures can make ingestion more complicated.

In this notebook, I learned how to:

- Create and inspect nested JSON
- Work with JSONL
- Use `JSONLoader`
- Use `jq_schema` to select specific data
- Create custom documents from nested JSON
- Preserve useful metadata
- Compare automatic loading with custom processing

The main flow I’m taking from this notebook is:

**JSON → Select Relevant Data → Structure the Content → Create Documents → Preserve Metadata**

The important lesson for me is that good RAG ingestion is not just about reading the file.

I also need to decide **what information should stay together** and **how that information should be represented for retrieval**.
